# 02 · Pilot-Guided, Mask-Aware Budgeted RePaint

**Research question.** With the denoiser-call budget of fixed RePaint held constant, does spending RePaint's extra resampling cycles where a pilot trajectory looks *unstable* in and around the hole give better inpainting than spending them uniformly?

This is a **hypothesis**. The code below is a tested implementation; whether it improves quality is an empirical question that only paired measurements on enough images can answer. "Pilot-Guided, Mask-Aware Budgeted RePaint" is a descriptive name, not a claim of novelty. Adaptive allocation of diffusion compute has been studied before (Chen et al., UAI 2024; Tang et al., ECCV 2024; Zhang et al., AAAI 2025; Li et al., arXiv 2412.05780). See `Docs/budget_aware_repaint_methods.md` for what is borrowed and what is not.

**Methods compared on the same image, mask, checkpoint, initial latent and noise seed**

| Method | What it does | Denoiser calls (T=250, j=10) |
|---|---|---|
| `fixed` | Paper schedule, r = 10 → 9 extra cycles at each of the 24 anchors (216 cycles) | 2,410 |
| `no_resampling` | r = 1, monotone 250 steps (lower-cost reference, **not** budget-matched) | 250 |
| `uniform` | Pilot (charged, unused) + 250 base + 191 cycles split evenly (8 or 7 per anchor) | 250 + 2,160 = 2,410 |
| `adaptive` | Pilot + 250 base + 191 cycles split by the pilot instability score | 250 + 2,160 = 2,410 |

The primary comparison is **adaptive vs uniform** (identical NFE and identical pilot cost; only the allocation differs). **adaptive vs fixed** asks whether the method pays for its pilot.

### Allocation rule

For each legal jump anchor *a* ∈ {0, 10, …, 230} (the keys of the official `get_schedule_jump` dictionary), the pilot gives clean-image estimates x̂₀ from the reverse calls that arrive at *a* (level *a*+1) and at *a*+j (level *a*+j+1). With D = x̂₀(*a*+1) − x̂₀(*a*+j+1), M the missing mask, B a band of `boundary_band_px` pixels on both sides of the seam and K the known region:

d_R(a) = √( Σ R·D² / (C·Σ R) ) for R ∈ {M, B, K}

n_hole = d_M / (d_K + ε),  n_band = d_B / (d_K + ε)  (`known_ratio` normalisation; raw values if there are no known pixels)

score(a) = n_hole(a)/Σ n_hole + λ · n_band(a)/Σ n_band

Each anchor first gets `min_extra_cycles`; the remaining cycles are split in proportion to the score with a `max_extra_cycles` cap (water-filling), floored, and the leftover cycles go to the largest fractional parts (ties: higher score, then lower anchor). The total is checked to equal the budget exactly. All-zero scores give a uniform split.

The score is a **heuristic trajectory-instability proxy**. It is not a calibrated uncertainty, and the pilot never sees the true missing pixels.

## 0 · Setup

In [10]:
# Locate the repository root and make `src` importable.
# The notebook lives in Notebooks/, but the project code is imported as the
# package `src` from the repository root, so the root must be on sys.path.
# Override with the REPAINT_PROJECT_ROOT environment variable (e.g. on Kaggle).
import os
import sys
from pathlib import Path

def _find_project_root():
    env = os.environ.get("REPAINT_PROJECT_ROOT")
    candidates = ([Path(env).expanduser()] if env else []) + [Path.cwd(), *Path.cwd().resolve().parents]
    for candidate in candidates:
        if (candidate / "src" / "repaint.py").is_file() and (candidate / "configs" / "config.yaml").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find the project root (needs src/repaint.py and configs/config.yaml). "
        "Start Jupyter inside the repository or set REPAINT_PROJECT_ROOT. Searched: "
        + ", ".join(map(str, candidates)))

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

Project root: /home/sarthak/programs/budget/Generative_Image_Impainting


In [11]:
import platform
import importlib
import torch

print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__, "| CUDA build:", torch.version.cuda)
print("CUDA usable :", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory / 2**30:.1f} GiB")
for name in ("diffusers", "torchvision", "torchmetrics", "matplotlib", "pandas", "yaml", "lpips"):
    try:
        module = importlib.import_module(name)
        print(f"{name:12s}:", getattr(module, "__version__", "installed"))
    except ImportError:
        print(f"{name:12s}: NOT INSTALLED" + ("" if name in ("torchmetrics", "lpips") else "  <-- required"))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device      :", DEVICE)

Python      : 3.10.20
PyTorch     : 2.14.0+cu130 | CUDA build: 13.0
CUDA usable : True
  GPU 0: NVIDIA GeForce RTX 3050 Laptop GPU, 3.7 GiB
diffusers   : 0.40.0
torchvision : 0.29.0+cu130
torchmetrics: 1.9.0
matplotlib  : 3.10.9
pandas      : 2.3.3
yaml        : 6.0.3
lpips       : NOT INSTALLED
Device      : cuda


In [12]:
from src.config import load_project_config, resolve_project_path

CONFIG_PATH = resolve_project_path("configs/config.yaml", PROJECT_ROOT, must_exist=True)
CFG = load_project_config(project_root=PROJECT_ROOT)
RCFG = CFG["repaint"]
ECFG = CFG.get("experiment", {})
MODEL_ID = CFG["model"]["pretrained_model_name"]
IMAGE_SIZE = int(CFG["data"]["image_size"])
T_STEPS = int(RCFG["num_inference_steps"])
JUMP_LENGTH = int(RCFG["jump_length"])
JUMP_N_SAMPLE = int(RCFG["jump_n_sample"])
MASK_RATIO = float(RCFG["mask_ratio"])
DATA_DIR = resolve_project_path(ECFG.get("data_dir", "data/celeba"), PROJECT_ROOT)
RESULTS_ROOT = resolve_project_path(ECFG.get("results_dir", "results_benchmarks"), PROJECT_ROOT)
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def list_images(data_dir):
    if not data_dir.exists():
        return []
    return sorted((p for p in data_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS),
                  key=lambda p: str(p).lower())

print("Config      :", CONFIG_PATH)
print("Model       :", MODEL_ID, "(Hugging Face id; downloaded on first load if not cached)")
print(f"RePaint     : T={T_STEPS}, jump_length={JUMP_LENGTH}, jump_n_sample={JUMP_N_SAMPLE}, mask_ratio={MASK_RATIO}")
print("Data dir    :", DATA_DIR, "| images found:", len(list_images(DATA_DIR)))
print("Results dir :", RESULTS_ROOT)
if not list_images(DATA_DIR):
    print("No images yet. Run `python scripts/download_celeba_sample.py` from the project root "
          "(needs the `datasets` package and internet), or put images in", DATA_DIR)

Config      : /home/sarthak/programs/budget/Generative_Image_Impainting/configs/config.yaml
Model       : google/ddpm-celebahq-256 (Hugging Face id; downloaded on first load if not cached)
RePaint     : T=250, jump_length=10, jump_n_sample=10, mask_ratio=0.35
Data dir    : /home/sarthak/programs/budget/Generative_Image_Impainting/data/celeba | images found: 20
Results dir : /home/sarthak/programs/budget/Generative_Image_Impainting/results_benchmarks


In [13]:
from src.adaptive_repaint import (CountingModel, allocate_cycles, plan_budget, run_comparison,
                                  uniform_allocation, anchor_comparison_levels)
from src.experiment_io import save_comparison
from src.repaint import eligible_jump_anchors

BCFG = CFG["budget_aware"]
LATENT_SEED = int(ECFG.get("latent_seed", RCFG["seed"]))
NOISE_SEED = int(ECFG.get("noise_seed", RCFG["seed"] + 1))
PILOT_NOISE_SEED = int(ECFG.get("pilot_noise_seed", NOISE_SEED))
BUDGET_KW = dict(
    reference_budget_nfe=BCFG.get("reference_budget_nfe"),
    min_extra_cycles=int(BCFG["min_extra_cycles"]), max_extra_cycles=BCFG.get("max_extra_cycles"),
    lambda_boundary=float(BCFG["lambda_boundary"]), boundary_band_px=int(BCFG["boundary_band_px"]),
    normalization=BCFG["normalization"], ratio_eps=float(BCFG["ratio_eps"]),
)
print(f"Seeds: latent (x_T) = {LATENT_SEED}, sampler noise = {NOISE_SEED}, pilot noise = {PILOT_NOISE_SEED}")
print("Budget-aware settings:", BUDGET_KW)

PLAN = plan_budget(T_STEPS, JUMP_LENGTH, JUMP_N_SAMPLE, BUDGET_KW["reference_budget_nfe"])
print(f"\nBudget plan (all counts from built schedules):")
print(f"  reference budget        {PLAN.reference_budget_nfe} NFE (fixed schedule = {PLAN.fixed_schedule_nfe})")
print(f"  pilot                   {PLAN.pilot_nfe}")
print(f"  base final trajectory   {PLAN.base_trajectory_nfe}")
print(f"  one extra cycle costs   {PLAN.nfe_per_cycle} NFE")
print(f"  extra cycles available  {PLAN.extra_cycles} over {PLAN.num_anchors} anchors (unspent {PLAN.unspent_nfe} NFE)")
print(f"  budgeted total          {PLAN.budgeted_total_nfe} NFE")
print("Uniform allocation:", uniform_allocation(PLAN.num_anchors, PLAN.extra_cycles))

Seeds: latent (x_T) = 42, sampler noise = 43, pilot noise = 43
Budget-aware settings: {'reference_budget_nfe': None, 'min_extra_cycles': 3, 'max_extra_cycles': 14, 'lambda_boundary': 0.5, 'boundary_band_px': 8, 'normalization': 'known_ratio', 'ratio_eps': 0.001}

Budget plan (all counts from built schedules):
  reference budget        2410 NFE (fixed schedule = 2410)
  pilot                   250
  base final trajectory   250
  one extra cycle costs   10 NFE
  extra cycles available  191 over 24 anchors (unspent 0 NFE)
  budgeted total          2410 NFE
Uniform allocation: [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7]


## 1 · Smoke test A — full-size schedule with a fake denoiser (no checkpoint)

Runs all four methods with the real T = 250 schedule on a 32×32 synthetic image and asserts every measured call count. Outputs are written to `results_benchmarks/budget_aware/smoke_fake/` so the figure section can be tried immediately. The images are meaningless.

In [14]:
# Test doubles for smoke tests that need no checkpoint or dataset.
# They only exercise the control flow, NFE accounting and file output; their
# images are meaningless.
from types import SimpleNamespace

class FakeScheduler:
    def __init__(self, num_train_timesteps=1000):
        self.betas = torch.linspace(1e-4, 0.02, num_train_timesteps)
        self.alphas_cumprod = torch.cumprod(1.0 - self.betas, dim=0)
        self.config = SimpleNamespace(num_train_timesteps=num_train_timesteps,
                                      prediction_type="epsilon", clip_sample=True)

class FakeDenoiser:
    def __call__(self, sample, timesteps):
        t = timesteps.to(sample.dtype).view(-1, 1, 1, 1) / 1000.0
        return SimpleNamespace(sample=0.5 * torch.tanh(sample) + 0.1 * t * sample.roll(1, dims=-1))

def synthetic_image(size, seed=0):
    g = torch.Generator().manual_seed(seed)
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, size), torch.linspace(-1, 1, size), indexing="ij")
    base = torch.stack([xx, yy, xx * yy]) * 0.8
    return (base + 0.05 * torch.randn(3, size, size, generator=g)).clamp(-1, 1).unsqueeze(0)

In [15]:
import time
x_fake = synthetic_image(32, seed=0)
m_fake = torch.zeros(1, 1, 32, 32); m_fake[..., 10:22, 8:24] = 1.0
t0 = time.perf_counter()
fake_result = run_comparison(CountingModel(FakeDenoiser()), FakeScheduler(), x_fake, m_fake,
                             T_STEPS, JUMP_LENGTH, JUMP_N_SAMPLE, LATENT_SEED, NOISE_SEED, PILOT_NOISE_SEED,
                             **BUDGET_KW)
M = fake_result.methods
assert M["fixed"].total_nfe == PLAN.fixed_schedule_nfe
assert M["no_resampling"].total_nfe == PLAN.base_trajectory_nfe
assert M["uniform"].total_nfe == M["adaptive"].total_nfe == PLAN.budgeted_total_nfe
assert sum(M["adaptive"].allocation.values()) == sum(M["uniform"].allocation.values()) == PLAN.extra_cycles
print(f"All NFE checks passed in {time.perf_counter() - t0:.1f} s")
SMOKE_FAKE_DIR = RESULTS_ROOT / "budget_aware" / "smoke_fake" / "synthetic_32px"
save_comparison(fake_result, x_fake, m_fake, SMOKE_FAKE_DIR,
                {"model_identifier": "FakeDenoiser (test double)", "image_id": "synthetic_32px",
                 "mask_type": "rectangle", "run_kind": "smoke_fake"}, compute_ssim=False, boundary_width=2)
print("Saved", SMOKE_FAKE_DIR)

[fixed] NFE=2410 (expected 2410), wall=1.2s
[no_resampling] NFE=250 (expected 250), wall=0.1s
[pilot] NFE=250, wall=0.1s
[uniform] NFE pilot 250 + final 2160 = 2410, wall=0.3s
[adaptive] NFE pilot 250 + final 2160 = 2410, wall=0.4s
All NFE checks passed in 2.2 s
Saved /home/sarthak/programs/budget/Generative_Image_Impainting/results_benchmarks/budget_aware/smoke_fake/synthetic_32px


## 2 · Smoke test B — real checkpoint, one image, reduced schedule

Loads `google/ddpm-celebahq-256` with the project loader and runs all four methods with T = 50, j = 5, r = 3 (fixed: 140 calls; budgeted: 50 pilot + 50 base + 8 cycles × 5 = 140). Per-anchor limits are relaxed (0–4) because there are only 8 cycles for 9 anchors. This verifies the wiring and call counts on the real model; it is far too small for quality conclusions.

In [16]:
RUN_REAL_MODEL_SMOKE = True   # set False if the checkpoint or data are not available
SMOKE = dict(num_inference_steps=50, jump_length=5, jump_n_sample=3)
SMOKE_KW = dict(BUDGET_KW, reference_budget_nfe=None, min_extra_cycles=0, max_extra_cycles=4)

from src.dataset import load_image
from src.mask import create_center_mask
from src.model import load_pretrained_ddpm

MODEL = SCHEDULER = None
SMOKE_REAL_DIR = None
if RUN_REAL_MODEL_SMOKE:
    images = list_images(DATA_DIR)
    if not images:
        raise FileNotFoundError(f"No images under {DATA_DIR}. See the message printed in §0.")
    MODEL, SCHEDULER, DEVICE = load_pretrained_ddpm(MODEL_ID, device=DEVICE)
    x0 = load_image(str(images[0]), IMAGE_SIZE).to(DEVICE)
    mask = create_center_mask(IMAGE_SIZE, MASK_RATIO).to(DEVICE)
    smoke_result = run_comparison(CountingModel(MODEL), SCHEDULER, x0, mask,
                                  SMOKE["num_inference_steps"], SMOKE["jump_length"], SMOKE["jump_n_sample"],
                                  LATENT_SEED, NOISE_SEED, PILOT_NOISE_SEED, **SMOKE_KW)
    SMOKE_REAL_DIR = RESULTS_ROOT / "budget_aware" / "smoke_real" / images[0].stem
    save_comparison(smoke_result, x0, mask, SMOKE_REAL_DIR,
                    {"model_identifier": MODEL_ID, "image_id": str(images[0].relative_to(DATA_DIR)),
                     "mask_type": f"center_side_ratio_{MASK_RATIO:g}", "run_kind": "smoke_real",
                     "scheduler_config": dict(SCHEDULER.config)})
    print("Allocation (adaptive):", smoke_result.methods["adaptive"].allocation)
    print("Saved", SMOKE_REAL_DIR)
else:
    print("Real-model smoke test skipped.")

An error occurred while trying to fetch google/ddpm-celebahq-256: google/ddpm-celebahq-256 does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


[fixed] NFE=140 (expected 140), wall=17.5s
[no_resampling] NFE=50 (expected 50), wall=6.2s
[pilot] NFE=50, wall=6.3s
[uniform] NFE pilot 50 + final 90 = 140, wall=11.2s
[adaptive] NFE pilot 50 + final 90 = 140, wall=11.2s
Allocation (adaptive): {0: 1, 5: 1, 10: 1, 15: 1, 20: 1, 25: 0, 30: 1, 35: 1, 40: 1}
Saved /home/sarthak/programs/budget/Generative_Image_Impainting/results_benchmarks/budget_aware/smoke_real/celeba_000


## 3 · Full experiment (explicit opt-in)

Paper schedule from `configs/config.yaml` (T = 250, j = 10, r = 10 → 2,410 NFE per method except `no_resampling`), first `evaluation.max_images` images, centre mask. Every image gets the same `LATENT_SEED`/`NOISE_SEED` across methods (pairing is per image). Per image this costs about 2,410 × 3 + 250 × 2 ≈ 7,700 UNet calls; on one T4 budget hours, not minutes.

Results go to `results_benchmarks/budget_aware/full/<image>/`. Re-running skips images whose `run.json` already exists unless `OVERWRITE = True`.

In [ ]:
RUN_FULL_EXPERIMENT = True   # set True deliberately
OVERWRITE = False

FULL_DIR = RESULTS_ROOT / "budget_aware" / "full" / f"T{T_STEPS}_j{JUMP_LENGTH}_r{JUMP_N_SAMPLE}_mask{MASK_RATIO:g}_lat{LATENT_SEED}_noise{NOISE_SEED}"
if RUN_FULL_EXPERIMENT:
    images = list_images(DATA_DIR)
    max_images = CFG.get("evaluation", {}).get("max_images")
    images = images[: int(max_images)] if max_images is not None else images
    if not images:
        raise FileNotFoundError(f"No images under {DATA_DIR}")
    if MODEL is None:
        MODEL, SCHEDULER, DEVICE = load_pretrained_ddpm(MODEL_ID, device=DEVICE)
    mask = create_center_mask(IMAGE_SIZE, MASK_RATIO).to(DEVICE)
    for index, path in enumerate(images):
        out_dir = FULL_DIR / path.stem
        if (out_dir / "run.json").exists() and not OVERWRITE:
            print(f"[{index + 1}/{len(images)}] {path.name}: already done, skipping"); continue
        print(f"[{index + 1}/{len(images)}] {path.name}")
        x0 = load_image(str(path), IMAGE_SIZE).to(DEVICE)
        result = run_comparison(CountingModel(MODEL), SCHEDULER, x0, mask, T_STEPS, JUMP_LENGTH, JUMP_N_SAMPLE,
                                LATENT_SEED, NOISE_SEED, PILOT_NOISE_SEED, **BUDGET_KW)
        save_comparison(result, x0, mask, out_dir,
                        {"model_identifier": MODEL_ID, "image_id": str(path.relative_to(DATA_DIR)),
                         "mask_type": f"center_side_ratio_{MASK_RATIO:g}", "run_kind": "full",
                         "scheduler_config": dict(SCHEDULER.config)})
    print("Saved under", FULL_DIR)
else:
    print("Full experiment not run (RUN_FULL_EXPERIMENT = False).")

[1/5] celeba_000.png
[fixed] NFE=2410 (expected 2410), wall=302.3s
[no_resampling] NFE=250 (expected 250), wall=31.4s
[pilot] NFE=250, wall=31.4s
[uniform] NFE pilot 250 + final 2160 = 2410, wall=271.2s
[adaptive] NFE pilot 250 + final 2160 = 2410, wall=271.2s
[2/5] celeba_001.png
[fixed] NFE=2410 (expected 2410), wall=302.7s
[no_resampling] NFE=250 (expected 250), wall=31.4s
[pilot] NFE=250, wall=31.4s
[uniform] NFE pilot 250 + final 2160 = 2410, wall=271.2s


## 4 · Results and figures (from saved CSV/JSON only)

Everything below reads the files written by `save_comparison`, so figures can be regenerated without re-running diffusion. Pick which run to plot: the full experiment if it exists, else the real smoke test, else the fake smoke test (whose images are meaningless). Figures are saved to `<run folder>/figures/`.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

def run_dirs(parent):
    if parent is None or not Path(parent).exists():
        return []
    parent = Path(parent)
    return sorted(p.parent for p in parent.rglob("run.json"))

candidates = [("full", FULL_DIR), ("smoke_real", (SMOKE_REAL_DIR.parent if SMOKE_REAL_DIR else None)),
              ("smoke_fake", SMOKE_FAKE_DIR.parent)]
RUN_KIND, RUN_DIRS = next(((k, run_dirs(d)) for k, d in candidates if run_dirs(d)), (None, []))
if not RUN_DIRS:
    raise FileNotFoundError("No saved runs found; run §1, §2 or §3 first.")
FIG_DIR = RUN_DIRS[0].parent / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plotting '{RUN_KIND}' results from {len(RUN_DIRS)} image folder(s); figures -> {FIG_DIR}")
if RUN_KIND == "smoke_fake":
    print("NOTE: fake-model outputs. The plots only demonstrate the pipeline; they carry no quality information.")

METHODS = ["fixed", "no_resampling", "uniform", "adaptive"]
LABEL = {"fixed": "Fixed RePaint (r=10)", "no_resampling": "No resampling (r=1)",
         "uniform": "Uniform allocation", "adaptive": "Adaptive allocation"}
SHORT = {"fixed": "Fixed\nr=10", "no_resampling": "No resampling\nr=1", "uniform": "Uniform\nallocation",
         "adaptive": "Adaptive\nallocation"}
COLOR = {"fixed": "#2a78d6", "no_resampling": "#eb6834", "uniform": "#1baf7a", "adaptive": "#eda100"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight", "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
                     "font.size": 9})

metrics = pd.concat([pd.read_csv(d / "metrics.csv") for d in RUN_DIRS], ignore_index=True)
alloc = pd.concat([pd.read_csv(d / "allocation.csv") for d in RUN_DIRS], ignore_index=True)
scores = pd.concat([pd.read_csv(d / "scores.csv") for d in RUN_DIRS if (d / "scores.csv").exists()], ignore_index=True)
runs = [json.loads((d / "run.json").read_text()) for d in RUN_DIRS]
metrics.to_csv(FIG_DIR.parent / "all_metrics.csv", index=False)
print("Images:", metrics["image_id"].nunique(), "| model:", runs[0]["model_identifier"])
print("Plan:", runs[0]["plan"])

### Comparison table

In [ ]:
COLS = ["total_nfe", "pilot_nfe", "final_nfe", "extra_cycles_total", "total_wall_time_s",
        "missing_psnr_db", "missing_mae", "boundary_inner_psnr_db", "composite_full_psnr_db",
        "composite_full_ssim", "lpips_composite", "raw_known_mae"]
COLS = [c for c in COLS if c in metrics and metrics[c].notna().any()]
table = metrics.groupby("method")[COLS].mean().reindex([m for m in METHODS if m in set(metrics.method)])
table.insert(0, "n_images", metrics.groupby("method").size())
display(table.round(4))
table.to_csv(FIG_DIR.parent / "comparison_table.csv")

# Paired per-image differences (same image, mask, x_T and noise seed).
QUALITY = [c for c in ["missing_psnr_db", "boundary_inner_psnr_db", "composite_full_psnr_db",
                       "composite_full_ssim", "lpips_composite"] if c in COLS]
wide = metrics.pivot_table(index="image_id", columns="method", values=QUALITY)
rows = []
for a, b in [("adaptive", "uniform"), ("adaptive", "fixed"), ("uniform", "fixed")]:
    if a in set(metrics.method) and b in set(metrics.method):
        for q in QUALITY:
            d = (wide[(q, a)] - wide[(q, b)]).dropna()
            better = (d < 0).sum() if q == "lpips_composite" else (d > 0).sum()
            rows.append({"comparison": f"{a} − {b}", "metric": q, "n_images": len(d), "mean_diff": d.mean(),
                         "std_diff": d.std(ddof=1) if len(d) > 1 else np.nan,
                         "images_where_first_is_better": int(better)})
paired = pd.DataFrame(rows)
display(paired.round(4))
paired.to_csv(FIG_DIR.parent / "paired_differences.csv", index=False)
print("With few images these are descriptive only; do not read them as significant improvements.")

### Figure 1 · Image panel (raw sampler outputs, then composites)

In [ ]:
def show_panel(run_dir, fname):
    img = lambda name: Image.open(run_dir / "images" / name)
    shown = [m for m in ["fixed", "uniform", "adaptive"] if (run_dir / "images" / f"{m}_raw.png").exists()]
    cols = 3 + len(shown)
    fig, axes = plt.subplots(2, cols, figsize=(2.3 * cols, 5.4))
    first = [("Original", img("original.png"), None), ("Mask (white = missing)", img("mask_missing_white.png"), "gray"),
             ("Masked input", img("masked_input.png"), None)]
    for c, (title, im, cmap) in enumerate(first):
        axes[0, c].imshow(im, cmap=cmap); axes[0, c].set_title(title)
        axes[1, c].axis("off")
    for c, m in enumerate(shown, start=3):
        axes[0, c].imshow(img(f"{m}_raw.png")); axes[0, c].set_title(f"{LABEL[m]}\nraw")
        axes[1, c].imshow(img(f"{m}_composite.png"))
    for a in axes.ravel():
        a.axis("off")
    run = json.loads((run_dir / "run.json").read_text())
    fig.suptitle(f"{run['image_id']} · {run['mask_type']} · x_T seed {run['settings']['latent_seed']}, "
                 f"noise seed {run['settings']['noise_seed']} · {run['plan']['reference_budget_nfe']} NFE")
    fig.text(0.5, 0.01, "Top: raw sampler output. Bottom: composite = raw output with known pixels restored.",
             ha="center", fontsize=8)
    plt.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(FIG_DIR / fname); plt.show()

for d in RUN_DIRS[:3]:
    show_panel(d, f"fig1_panel_{d.name}.png")

### Figure 2 · Quality metrics by method (bars = mean over images, dots = individual images)

In [ ]:
panels = [("missing_psnr_db", "Missing-region PSNR (dB) ↑"), ("boundary_inner_psnr_db", "Inner boundary band PSNR (dB) ↑"),
          ("composite_full_ssim", "Composite full-image SSIM ↑"), ("lpips_composite", "Composite LPIPS ↓")]
panels = [p for p in panels if p[0] in metrics and metrics[p[0]].notna().any()]
present = [m for m in METHODS if m in set(metrics.method)]
fig, axes = plt.subplots(1, len(panels), figsize=(3.3 * len(panels), 3.4), squeeze=False)
for ax, (col, title) in zip(axes[0], panels):
    for i, m in enumerate(present):
        vals = metrics.loc[metrics.method == m, col].dropna()
        ax.bar(i, vals.mean(), width=0.7, color=COLOR[m], zorder=2)
        ax.scatter(np.full(len(vals), i), vals, s=12, color="white", edgecolor="#222", linewidth=0.6, zorder=3)
    ax.set_xticks(range(len(present)), [SHORT[m] for m in present], fontsize=7)
    ax.set_title(title); ax.grid(axis="x", visible=False)
    lo = metrics[col].min(); hi = metrics[col].max(); pad = (hi - lo) * 0.3 + 1e-6
    ax.set_ylim(lo - pad, hi + pad)
fig.suptitle(f"{metrics.image_id.nunique()} image(s) · y-axes zoomed to the data range")
plt.tight_layout(); fig.savefig(FIG_DIR / "fig2_metrics_by_method.png"); plt.show()

### Figure 3 · Pilot instability scores per anchor

In [ ]:
if len(scores):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
    for image_id, s in scores.groupby("image_id"):
        s = s.sort_values("anchor")
        axes[0].plot(s["native_timestep"], s["score"], marker="o", ms=3, lw=1.5, label=str(image_id)[:24])
        axes[1].plot(s["native_timestep"], s["d_hole"], lw=1.2, color="#2a78d6", alpha=0.8)
        axes[1].plot(s["native_timestep"], s["d_known"], lw=1.2, color="#eb6834", alpha=0.8, ls="--")
    axes[0].set_xlabel("Anchor native timestep t (noisier →)"); axes[0].set_ylabel("score = hole share + λ·band share")
    axes[0].set_title(f"Combined score (normalisation: {runs[0]['score_settings'].get('normalization')})")
    if scores.image_id.nunique() <= 6: axes[0].legend(frameon=False, fontsize=7)
    axes[1].set_yscale("log"); axes[1].set_xlabel("Anchor native timestep t")
    axes[1].set_ylabel("RMS change of x̂₀ across the window ([-1,1] units)")
    axes[1].set_title("Raw drift: hole (solid) vs known region (dashed)")
    for ax in axes: ax.invert_xaxis()
    plt.tight_layout(); fig.savefig(FIG_DIR / "fig3_pilot_scores.png"); plt.show()
else:
    print("No pilot scores in these runs.")

### Figure 4 · Extra resampling cycles per anchor

In [ ]:
present_alloc = [m for m in ["fixed", "uniform", "adaptive"] if f"{m}_extra_cycles" in alloc]
mean_alloc = alloc.groupby("anchor")[[f"{m}_extra_cycles" for m in present_alloc]].mean()
x = np.arange(len(mean_alloc)); w = 0.8 / len(present_alloc)
fig, ax = plt.subplots(figsize=(11, 3.4))
for i, m in enumerate(present_alloc):
    ax.bar(x + (i - (len(present_alloc) - 1) / 2) * w, mean_alloc[f"{m}_extra_cycles"], w * 0.9,
           color=COLOR[m], label=f"{LABEL[m]} (total {int(alloc.groupby('image_id')[f'{m}_extra_cycles'].sum().iloc[0])})")
if "adaptive_extra_cycles" in alloc and alloc.image_id.nunique() > 1:
    for _, a in alloc.groupby("image_id"):
        ax.scatter(x + (len(present_alloc) - 1) / 2 * w, a.sort_values("anchor")["adaptive_extra_cycles"], s=6, color="#333", zorder=3)
ax.set_xticks(x, [str(a) for a in mean_alloc.index], fontsize=7)
ax.set_xlabel("Anchor (respaced level; diffusion runs right → left)"); ax.set_ylabel("Extra cycles")
ax.invert_xaxis(); ax.grid(axis="x", visible=False)
ax.legend(frameon=False, fontsize=8, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0))
ax.set_title("Allocation of extra cycles (adaptive: mean over images, dots = individual images)", pad=24)
plt.tight_layout(); fig.savefig(FIG_DIR / "fig4_allocation.png"); plt.show()

### Figure 5 · Compute: denoiser calls and wall-clock time (two separate scales)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
present = [m for m in METHODS if m in set(metrics.method)]
agg = metrics.groupby("method")[["pilot_nfe", "final_nfe", "pilot_wall_time_s", "final_wall_time_s"]].mean().reindex(present)
for i, m in enumerate(present):
    axes[0].bar(i, agg.loc[m, "final_nfe"], color=COLOR[m], width=0.7)
    axes[0].bar(i, agg.loc[m, "pilot_nfe"], bottom=agg.loc[m, "final_nfe"], color=COLOR[m], width=0.7, hatch="//",
                edgecolor="white", alpha=0.6)
    axes[0].text(i, agg.loc[m, "final_nfe"] + agg.loc[m, "pilot_nfe"], f"{int(agg.loc[m, 'final_nfe'] + agg.loc[m, 'pilot_nfe'])}",
                 ha="center", va="bottom", fontsize=8)
    axes[1].bar(i, agg.loc[m, "final_wall_time_s"], color=COLOR[m], width=0.7)
    axes[1].bar(i, agg.loc[m, "pilot_wall_time_s"], bottom=agg.loc[m, "final_wall_time_s"], color=COLOR[m], width=0.7,
                hatch="//", edgecolor="white", alpha=0.6)
for ax, title in zip(axes, ["Denoiser evaluations (NFE) per image", "Wall-clock time per image (s)"]):
    ax.set_xticks(range(len(present)), [SHORT[m] for m in present], fontsize=7)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.12)
    ax.set_title(title + "\nhatched = pilot + scoring"); ax.grid(axis="x", visible=False)
plt.tight_layout(); fig.savefig(FIG_DIR / "fig5_compute.png"); plt.show()
print("Environment of first run:", runs[0]["environment"])

### Figure 6 · Missing region and boundary, with absolute error maps

In [ ]:
def show_hole(run_dir, fname, pad=12):
    mask = np.array(Image.open(run_dir / "images" / "mask_missing_white.png")) > 127
    ys, xs = np.where(mask)
    if len(ys) == 0:
        print("Empty mask; nothing to crop"); return
    y0, y1 = max(ys.min() - pad, 0), min(ys.max() + pad + 1, mask.shape[0])
    x0_, x1 = max(xs.min() - pad, 0), min(xs.max() + pad + 1, mask.shape[1])
    gt = np.asarray(Image.open(run_dir / "images" / "original.png"), dtype=np.float32) / 255
    shown = [m for m in ["fixed", "uniform", "adaptive"] if (run_dir / "images" / f"{m}_composite.png").exists()]
    comps = {m: np.asarray(Image.open(run_dir / "images" / f"{m}_composite.png"), dtype=np.float32) / 255 for m in shown}
    errs = {m: np.abs(c - gt).mean(-1) for m, c in comps.items()}
    vmax = max(float(e[y0:y1, x0_:x1].max()) for e in errs.values()) or 1.0
    fig, axes = plt.subplots(2, len(shown) + 1, figsize=(2.6 * (len(shown) + 1), 5.2))
    crop = lambda a: a[y0:y1, x0_:x1]
    axes[0, 0].imshow(crop(gt)); axes[0, 0].set_title("Ground truth (crop)")
    axes[1, 0].axis("off")
    for c, m in enumerate(shown, start=1):
        axes[0, c].imshow(crop(comps[m])); axes[0, c].set_title(LABEL[m])
        im = axes[1, c].imshow(crop(errs[m]), cmap="Blues", vmin=0, vmax=vmax)
        axes[1, c].set_title("|error|", fontsize=8)
    for a in axes.ravel():
        a.contour(crop(mask), levels=[0.5], colors="#eb6834", linewidths=0.8) if a.images else None
        a.set_xticks([]); a.set_yticks([]); a.grid(False)
    fig.colorbar(im, ax=axes[1, 1:].tolist(), shrink=0.8, label="absolute error [0, 1]")
    fig.suptitle(f"{json.loads((run_dir / 'run.json').read_text())['image_id']} · orange line = hole boundary · "
                 "bottom: |composite − ground truth|, mean over RGB")
    fig.savefig(FIG_DIR / fname); plt.show()

for d in RUN_DIRS[:3]:
    show_hole(d, f"fig6_hole_{d.name}.png")
print("Figures saved:", sorted(p.name for p in FIG_DIR.glob("*.png")))

## 5 · Reading the results

* **Implementation checks** (call counts, exact cycle totals, legal anchors, identical transitions) are asserted in code and in `tests/`. Passing them shows the comparison is *fair*, not that the method *works*.
* **adaptive − uniform** isolates the allocation rule. **adaptive − fixed** includes the cost of the pilot (adaptive has 191 extra cycles vs 216 for fixed).
* Trajectories share x_T and the noise seed, but diverge after the first anchor where the schedules differ. Differences on a handful of images can be noise; use many images (the paper used 100 per mask) and report paired differences with their spread before claiming an improvement.
* Missing-region PSNR rewards closeness to one particular ground truth. RePaint is stochastic and can produce plausible but different completions, so also look at perceptual metrics (LPIPS, if installed) and the images.
* The model is `google/ddpm-celebahq-256`, not the paper's checkpoint; absolute numbers are not comparable to the paper's Table 1.